In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import numpy as np
import datetime
import time

import sys
sys.path.append("src")

from data_generation import *
from neural_networks import *
from estimation import *

pd.options.mode.chained_assignment = None
os.chdir('Result_Tables/')

## RCL

In [2]:
seed_list = [7456,145609, 214341, 237639, 148649, 763490, 677253, 560404, 591830,
    84947,  26672, 184535, 874369,  39876, 738836, 100589, 812811,
    75122, 182935, 911677]

dg_list = [rcl] 
JKM_list = [[10,10,100],[5,10,100],[20,10,100],[10,10,20],[10,10,200]]
delta = 0.1 

hyper_params = {'hidden_size': 256, 'num_hidden_layers': 5, 'n_epochs': 2000, 'learning_rate': 0.0001}

for [J, K, M] in JKM_list:
    for dg in dg_list:
        for i in range(len(seed_list[1:3])):
            print(J, K, M, i, 'started', datetime.datetime.now())
            seed = seed_list[i]
            params = [np.random.normal(0,1,K+2) / (K * 2), np.ones(K+2)]
            params[0][-1] = -1
            full_one_iteration(J, M, K, seed, dg, params, hyper_params, delta)   ### this line runs for about 5-10 min 
            print(J, K, M, i, 'finished', datetime.datetime.now())
        report(J, K, M, seed_list, dg)

10 10 100 0 started 2026-03-09 15:32:12.571492
Data is generated.
Main model is trained.
RCL is estimated.
MNL is estimated.
NP is estimated.
Prediction errors are calculated.
Elasticity errors are calculated.
10 10 100 0 finished 2026-03-09 15:36:37.616189
10 10 100 1 started 2026-03-09 15:36:37.616230
Data is generated.
Main model is trained.
RCL is estimated.
MNL is estimated.
NP is estimated.
Prediction errors are calculated.
Elasticity errors are calculated.
10 10 100 1 finished 2026-03-09 15:41:06.190972
5 10 100 0 started 2026-03-09 15:41:11.170913
Data is generated.
Main model is trained.
RCL is estimated.
MNL is estimated.
NP is estimated.
Prediction errors are calculated.
Elasticity errors are calculated.
5 10 100 0 finished 2026-03-09 15:45:55.769528
5 10 100 1 started 2026-03-09 15:45:55.769568
Data is generated.
Main model is trained.
RCL is estimated.
MNL is estimated.
NP is estimated.
Prediction errors are calculated.
Elasticity errors are calculated.
5 10 100 1 finished

In [4]:
## this step reads the middle tables saved from the above cell
JKM_dg_list = [[10,10,100,rcl],[5,10,100,rcl],[20,10,100,rcl],[10,10,20,rcl],[10,10,200,rcl]]
elas_median_df, elas_mae_df, elas_rmse_df, error_df  = output(JKM_dg_list)

In [5]:
error_rslt = error_df.groupby(['dg','J','M','K'])[['mae_deep','mae_mnl','mae_rcl','mae_single','mae_random',
                                                  'mse_deep','mse_mnl','mse_rcl','mse_single','mse_random']].mean().reset_index()
error_rslt[['rmse_deep','rmse_mnl','rmse_rcl','rmse_single','rmse_random']] = np.sqrt(error_rslt[['mse_deep','mse_mnl','mse_rcl','mse_single','mse_random']])
error_rslt['obs'] = error_rslt['M'] * error_rslt['J'] *  0.2
error_rslt = error_rslt[[ 'dg', 'J', 'M', 'K', 
                             'mae_deep', 'rmse_deep', 
                             'mae_mnl', 'rmse_mnl', 
                             'mae_rcl', 'rmse_rcl',  
                             'mae_single', 'rmse_single', 
                             'mae_random','rmse_random',
                             'obs']]
error_rslt

,dg,J,M,K,mae_deep,rmse_deep,mae_mnl,rmse_mnl,mae_rcl,rmse_rcl,mae_single,rmse_single,mae_random,rmse_random,obs
0,rcl,5,100,10,0.024321,0.054382,0.031421,0.067499,0.003405,0.018461,0.047001,0.090634,0.055095,0.096286,100.0
1,rcl,10,20,10,0.021112,0.055104,0.027538,0.060609,0.003433,0.016867,0.055351,0.095052,0.042823,0.078100,40.0
2,rcl,10,100,10,0.017724,0.047309,0.026874,0.059930,0.002531,0.016486,0.046952,0.087921,0.042275,0.080605,200.0
3,rcl,10,200,10,0.014884,0.044183,0.025859,0.058057,0.003285,0.018342,0.044274,0.084190,0.042149,0.079928,400.0
4,rcl,20,100,10,0.010306,0.035670,0.026533,0.055135,0.001871,0.013644,0.040138,0.078788,0.028438,0.061967,400.0


In [6]:
elas_rmse_df.columns = [col.replace('ae_', 'rmse_') if 'ae_' in col else col for col in elas_rmse_df.columns]
elas_mae_df.columns = [col.replace('ae_', 'mae_') if 'ae_' in col else col for col in elas_mae_df.columns]

elas_rslt = pd.merge(elas_mae_df, elas_rmse_df,
                     on = ['type','dg','J','M','K'])
elas_rslt['obs'] = (elas_rslt['M'] * elas_rslt['J'] * 20 * 0.8).astype('int')
elas_rslt.loc[elas_rslt.type == 'cross','obs'] = (elas_rslt['M'] * elas_rslt['J'] * 20 * (elas_rslt['J']-1) * 0.8).astype('int')

elas_rslt = elas_rslt[['type', 'dg', 'J', 'M', 'K', 
                             'mae_deep', 'rmse_deep', 
                             'mae_mnl', 'rmse_mnl', 
                             'mae_rcl', 'rmse_rcl',  
                             'mae_single', 'rmse_single', 
                             'obs']]
elas_rslt

,type,dg,J,M,K,mae_deep,rmse_deep,mae_mnl,rmse_mnl,mae_rcl,rmse_rcl,mae_single,rmse_single,obs
0,cross,rcl,10,100,10,0.038723,0.055955,0.038540,0.050909,0.004921,0.008596,0.107917,0.146992,144000
1,self,rcl,10,100,10,0.210416,0.291323,0.433645,0.653900,0.039237,0.064019,0.330386,0.402929,16000
2,cross,rcl,5,100,10,0.042288,0.059850,0.057779,0.074584,0.004686,0.007292,0.086282,0.119538,32000
3,self,rcl,5,100,10,0.171667,0.239711,0.289471,0.433393,0.020299,0.031118,0.235892,0.315245,8000
4,cross,rcl,20,100,10,0.033912,0.051154,0.026631,0.037382,0.005125,0.010603,0.092367,0.122568,608000
5,self,rcl,20,100,10,0.256378,0.374322,1.892881,2.539378,0.087866,0.146900,0.472485,0.559217,32000
6,cross,rcl,10,20,10,0.038236,0.055714,0.039818,0.052800,0.008013,0.015186,0.110140,0.135364,28800
7,self,rcl,10,20,10,0.278889,0.401150,0.402188,0.610224,0.076757,0.130978,0.381791,0.447052,3200
8,cross,rcl,10,200,10,0.032401,0.048331,0.041117,0.053784,0.005819,0.010098,0.107050,0.150852,288000
9,self,rcl,10,200,10,0.178465,0.254060,0.430128,0.646501,0.042437,0.074205,0.317016,0.423953,32000


## MNL

In [9]:
seed_list = [7456,145609, 214341, 237639, 148649, 763490, 677253, 560404, 591830,
        84947,  26672, 184535, 874369,  39876, 738836, 100589, 812811,
        75122, 182935, 911677]
    
dg_list = [mnl_choice] 
JKM_list = [[10,10,100],[5,10,100],[20,10,100],
            [10,10,20],[10,10,200]]
delta = 0.1 

hyper_params = {'hidden_size': 256, 'num_hidden_layers': 5, 'n_epochs': 2000, 'learning_rate': 0.0001}

for [J, K, M] in JKM_list:
    for dg in dg_list:
        for i in range(len(seed_list[1:3])):
            seed = seed_list[i]
            params =  np.ones(K+2)
            params[-1] = -1
            full_one_iteration(J, M, K, seed, dg, params, hyper_params, delta)
            print(J, K, M, i, 'finished', datetime.datetime.now())
            

Data is generated.
Main model is trained.
RCL is estimated.
MNL is estimated.
NP is estimated.
Prediction errors are calculated.
Elasticity errors are calculated.
10 10 100 0 finished 2026-03-09 17:34:13.858858
Data is generated.
Main model is trained.
RCL is estimated.
MNL is estimated.
NP is estimated.
Prediction errors are calculated.
Elasticity errors are calculated.
10 10 100 1 finished 2026-03-09 17:38:37.600273
Data is generated.
Main model is trained.
RCL is estimated.
MNL is estimated.
NP is estimated.
Prediction errors are calculated.
Elasticity errors are calculated.
5 10 100 0 finished 2026-03-09 17:42:36.240696
Data is generated.
Main model is trained.
RCL is estimated.
MNL is estimated.
NP is estimated.
Prediction errors are calculated.
Elasticity errors are calculated.
5 10 100 1 finished 2026-03-09 17:46:26.388129
Data is generated.
Main model is trained.
RCL is estimated.
MNL is estimated.
NP is estimated.
Prediction errors are calculated.
Elasticity errors are calcula

In [11]:
## this step reads the middle tables saved from the above cell
#os.chdir('Result_Tables/')
JKM_dg_list = [[10,10,100,mnl_choice],[5,10,100,mnl_choice],[20,10,100,mnl_choice],[10,10,20,mnl_choice],[10,10,200,mnl_choice]]
elas_median_df, elas_mae_df, elas_rmse_df, error_df  = output(JKM_dg_list)

In [12]:
error_rslt = error_df.groupby(['dg','J','M','K'])[['mae_deep','mae_mnl','mae_rcl','mae_single','mae_random',
                                                  'mse_deep','mse_mnl','mse_rcl','mse_single','mse_random']].mean().reset_index()
error_rslt[['rmse_deep','rmse_mnl','rmse_rcl','rmse_single','rmse_random']] = np.sqrt(error_rslt[['mse_deep','mse_mnl','mse_rcl','mse_single','mse_random']])
error_rslt['obs'] = error_rslt['M'] * error_rslt['J'] *  0.2
error_rslt = error_rslt[[ 'dg', 'J', 'M', 'K', 
                             'mae_deep', 'rmse_deep', 
                             'mae_mnl', 'rmse_mnl', 
                             'mae_rcl', 'rmse_rcl',  
                             'mae_single', 'rmse_single', 
                             'mae_random','rmse_random',
                             'obs']]
error_rslt

,dg,J,M,K,mae_deep,rmse_deep,mae_mnl,rmse_mnl,mae_rcl,rmse_rcl,mae_single,rmse_single,mae_random,rmse_random,obs
0,mnl_choice,5,100,10,0.053413,0.109403,0.007836,0.013680,0.008150,0.013824,0.126880,0.220438,0.231156,0.294748,100.0
1,mnl_choice,10,20,10,0.058524,0.135302,0.004014,0.010411,0.008852,0.019799,0.112946,0.221431,0.136525,0.210474,40.0
2,mnl_choice,10,100,10,0.033269,0.083859,0.004428,0.010891,0.002580,0.005302,0.118078,0.238674,0.142214,0.221929,200.0
3,mnl_choice,10,200,10,0.030694,0.076849,0.003194,0.008081,0.003424,0.006942,0.109631,0.225058,0.141646,0.220110,400.0
4,mnl_choice,20,100,10,0.019371,0.059114,0.001514,0.003976,0.002297,0.005402,0.070700,0.172281,0.076794,0.150314,400.0


In [13]:
elas_rmse_df.columns = [col.replace('ae_', 'rmse_') if 'ae_' in col else col for col in elas_rmse_df.columns]
elas_mae_df.columns = [col.replace('ae_', 'mae_') if 'ae_' in col else col for col in elas_mae_df.columns]

elas_rslt = pd.merge(elas_mae_df, elas_rmse_df,
                     on = ['type','dg','J','M','K'])
elas_rslt['obs'] = (elas_rslt['M'] * elas_rslt['J'] * 20 * 0.8).astype('int')
elas_rslt.loc[elas_rslt.type == 'cross','obs'] = (elas_rslt['M'] * elas_rslt['J'] * 20 * (elas_rslt['J']-1) * 0.8).astype('int')

elas_rslt = elas_rslt[['type', 'dg', 'J', 'M', 'K', 
                             'mae_deep', 'rmse_deep', 
                             'mae_mnl', 'rmse_mnl', 
                             'mae_rcl', 'rmse_rcl',  
                             'mae_single', 'rmse_single', 
                             'obs']]
elas_rslt

,type,dg,J,M,K,mae_deep,rmse_deep,mae_mnl,rmse_mnl,mae_rcl,rmse_rcl,mae_single,rmse_single,obs
0,cross,mnl_choice,10,100,10,0.203392,0.610397,0.094420,0.514250,0.096950,0.514300,0.375353,0.798429,144000
1,self,mnl_choice,10,100,10,0.749541,1.444984,0.684992,1.385756,0.692198,1.388443,1.302108,1.982585,16000
2,cross,mnl_choice,5,100,10,0.299972,0.811498,0.150338,0.678144,0.159057,0.681442,0.433846,0.976686,32000
3,self,mnl_choice,5,100,10,0.591715,1.190699,0.506116,1.125237,0.541630,1.142993,0.760321,1.317710,8000
4,cross,mnl_choice,20,100,10,0.117532,0.466503,0.059540,0.415079,0.062195,0.415703,0.233549,0.565711,608000
5,self,mnl_choice,20,100,10,0.902872,1.670628,0.881588,1.620559,0.902404,1.625780,1.498232,2.316448,32000
6,cross,mnl_choice,10,20,10,0.197950,0.681959,0.091853,0.532632,0.106149,0.542601,0.321090,0.776352,28800
7,self,mnl_choice,10,20,10,0.746176,1.355548,0.641605,1.297462,0.704200,1.329818,1.350398,1.986911,3200
8,cross,mnl_choice,10,200,10,0.202421,0.607842,0.094880,0.514044,0.099824,0.515345,0.404497,0.816042,288000
9,self,mnl_choice,10,200,10,0.740116,1.416136,0.686092,1.384745,0.712648,1.393948,1.210355,1.897544,32000
